# Searching for similar books

Run after `data-exploration.ipynb`. First export the tagged descriptions, then optionally try semantic search. Run from the repository root with the local package installed.


In [1]:
from pathlib import Path

import pandas as pd

books = pd.read_csv("books_cleaned.csv", dtype={"isbn13": str})
books[["title", "tagged_description"]].head()

,title,tagged_description
0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,Spider's Web,9780002261982 A new 'Christie for Christmas' -...
2,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."


## Save one description per line

This export does not call OpenAI. Replace line breaks inside descriptions so each book stays on one line.


In [2]:
descriptions = []
for _, book in books.iterrows():
    description = " ".join(book["description"].split())
    descriptions.append(f"{book['isbn13']} {description}")

Path("tagged_description.txt").write_text(
    "\n".join(descriptions) + "\n", encoding="utf-8"
)
print(f"Saved {len(descriptions)} descriptions to tagged_description.txt")

Saved 5197 descriptions to tagged_description.txt


## Optional: build the search index

**The following cells send descriptions to OpenAI and make paid API calls.** Set `OPENAI_API_KEY` in the root `.env` first. Restarting this notebook and rebuilding the index embeds the descriptions again. You can stop here if you only need the export for the dashboard.


In [3]:
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

load_dotenv()
documents = []
for description in descriptions:
    documents.append(Document(page_content=description))

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
database = Chroma.from_documents(documents, embeddings)

## Try a search

Reuse the app's selection code so the results keep the similarity order and respect the requested number of books.


In [4]:
from book_recommender.recommendations import select_recommendations

catalogue = {}
for book in books.to_dict("records"):
    catalogue[book["isbn13"]] = book

query = "A book to teach children about nature"
search_results = database.similarity_search(query, k=min(10, len(books)))
recommendations = select_recommendations(catalogue, search_results, limit=10)
pd.DataFrame(recommendations)[["title", "authors", "description"]]

,title,authors,description
0,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Children will discover the exciting world of t...
1,Baby Einstein: Babies,Julie Aigner-Clark,"Introduce your babies to birds, cats, dogs, an..."
2,Baby Einstein: Dogs,Julie Aigner-Clark,"Introduce your baby to birds, cats, dogs, and ..."
3,Baby Einstein: Birds,Julie Aigner-Clark,"Introducing your baby to birds, cats, dogs, an..."
4,"Moo, Baa, la la La!",Sandra Boynton,Children will love joining in and imitating th...
5,Tree,David Burnie,Photographs and text explore the anatomy and l...
6,Baby Einstein: What Does Violet See? Raindrops...,Julie Aigner-Clark,A very special puddle sets Violet the mouse of...
7,The Sense of Wonder,Rachel Carson,"First published more than three decades ago, t..."
8,The Big Box,Toni Morrison;Slade Morrison,"In her first illustrated book for children, th..."
9,Judy Moody Saves the World!,Megan McDonald,When Judy Moody gets serious about protecting ...
